In [1]:
# ============================================================
# GENERATE SYNTHETIC WHATSAPP SCREENSHOTS FROM TEXT DATA
# Create training images without real screenshots!
# ============================================================

!pip install pillow numpy pandas matplotlib --quiet

from PIL import Image, ImageDraw, ImageFont
import pandas as pd
import numpy as np
import random
import os

def create_fake_whatsapp_chat(text, is_bullying, output_path):
    """
    Generate a fake WhatsApp screenshot from text
    """
    # WhatsApp colors
    BG_COLOR = (230, 228, 222)  # WhatsApp background
    BUBBLE_GREEN = (37, 211, 102)  # Sent message
    BUBBLE_WHITE = (255, 255, 255)  # Received message
    
    # Create image
    width, height = 800, 1000
    img = Image.new('RGB', (width, height), BG_COLOR)
    draw = ImageDraw.Draw(img)
    
    try:
        font = ImageFont.truetype("arial.ttf", 24)
        name_font = ImageFont.truetype("arial.ttf", 18)
    except:
        font = ImageFont.load_default()
        name_font = ImageFont.load_default()
    
    # Header
    draw.rectangle([0, 0, width, 80], fill=(7, 94, 84))
    draw.text((20, 25), "WhatsApp Chat", fill=(255,255,255), font=font)
    
    # Split text into messages
    sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 10]
    if len(sentences) < 2:
        sentences = text.split('\n')
    
    y_pos = 100
    is_sender = True
    
    for sentence in sentences[:8]:  # Max 8 messages
        if len(sentence) < 5:
            continue
            
        # Alternate between sender and receiver
        if is_sender:
            bubble_color = BUBBLE_GREEN
            x_start, x_end = 400, 750
        else:
            bubble_color = BUBBLE_WHITE
            x_start, x_end = 50, 400
        
        # Wrap text
        words = sentence.split()
        lines = []
        current = ""
        for word in words:
            if len(current + word) < 35:
                current += word + " "
            else:
                lines.append(current)
                current = word + " "
        lines.append(current)
        
        # Calculate bubble height
        bubble_height = len(lines) * 30 + 20
        
        # Draw bubble
        draw.rounded_rectangle(
            [x_start, y_pos, x_end, y_pos + bubble_height], 
            radius=15, 
            fill=bubble_color
        )
        
        # Draw text
        text_y = y_pos + 10
        for line in lines:
            text_color = (255, 255, 255) if is_sender else (0, 0, 0)
            draw.text((x_start + 15, text_y), line.strip(), fill=text_color, font=font)
            text_y += 28
        
        y_pos += bubble_height + 15
        is_sender = not is_sender
    
    # Save
    img.save(output_path)
    return output_path

# Generate dataset from your CSV
def generate_whatsapp_dataset(df, output_folder="whatsapp_synthetic", samples_per_class=100):
    """
    Create synthetic WhatsApp screenshots from your text data
    """
    os.makedirs(f"{output_folder}/bullying", exist_ok=True)
    os.makedirs(f"{output_folder}/safe", exist_ok=True)
    
    # Get bullying and safe texts
    bullying_texts = df[df['oh_label'] == 1]['cleaned_text'].tolist()
    safe_texts = df[df['oh_label'] == 0]['cleaned_text'].tolist()
    
    print(f"🎨 Generating {samples_per_class} synthetic screenshots per class...")
    
    # Generate bullying screenshots
    for i in range(min(samples_per_class, len(bullying_texts))):
        text = bullying_texts[i]
        create_fake_whatsapp_chat(text, True, f"{output_folder}/bullying/chat_{i}.png")
        if (i+1) % 20 == 0:
            print(f"   Generated {i+1} bullying screenshots")
    
    # Generate safe screenshots
    for i in range(min(samples_per_class, len(safe_texts))):
        text = safe_texts[i]
        create_fake_whatsapp_chat(text, False, f"{output_folder}/safe/chat_{i}.png")
        if (i+1) % 20 == 0:
            print(f"   Generated {i+1} safe screenshots")
    
    print(f"✅ Dataset created in ./{output_folder}/")
    print(f"   Bullying: {min(samples_per_class, len(bullying_texts))} images")
    print(f"   Safe: {min(samples_per_class, len(safe_texts))} images")

# Usage
df = pd.read_csv("./cyberbullying_data_labelled.csv")
generate_whatsapp_dataset(df, samples_per_class=200)


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


🎨 Generating 200 synthetic screenshots per class...
   Generated 20 bullying screenshots
   Generated 40 bullying screenshots
   Generated 60 bullying screenshots
   Generated 80 bullying screenshots
   Generated 100 bullying screenshots
   Generated 120 bullying screenshots
   Generated 140 bullying screenshots
   Generated 160 bullying screenshots
   Generated 180 bullying screenshots
   Generated 200 bullying screenshots
   Generated 20 safe screenshots
   Generated 40 safe screenshots
   Generated 60 safe screenshots
   Generated 80 safe screenshots
   Generated 100 safe screenshots
   Generated 120 safe screenshots
   Generated 140 safe screenshots
   Generated 160 safe screenshots
   Generated 180 safe screenshots
   Generated 200 safe screenshots
✅ Dataset created in ./whatsapp_synthetic/
   Bullying: 200 images
   Safe: 200 images
